# Mood classification of short Russian texts

Three text representations compared on the same data and the same evaluation protocol.

1. TF-IDF over character n-grams
2. `rubert-tiny2`, `[CLS]` vector
3. `rubert-tiny2`, mean pooling over tokens

Classifier, splitting scheme and metric are identical in all three, so any difference comes from the representation.

In [ ]:
!pip install -q transformers

In [ ]:
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel

## Data

Each row is one line of a free-form description of a track. `track_id` marks which track the line came from: lines of one track were written in one sitting, so they must never be split across train and test.

In [ ]:
df = pd.read_csv('sample.csv', sep=';')
df['label'] = df['label'].str.strip().str.lower()

X = df['text']
y = df['label']
groups = df['track_id']

print(df.shape)
print(y.value_counts())

## Evaluation protocol

`GroupKFold` grouped by `track_id`. Macro-F1 is the primary metric: the classes are imbalanced, and accuracy would reward ignoring the small ones.

In [ ]:
cv = GroupKFold(n_splits=5)
clf = LogisticRegression(max_iter=1000, class_weight='balanced')

## Experiment 1. TF-IDF baseline

Character n-grams rather than words: Russian is inflected, so `тоска` and `тоскливо` share character sequences but are different word tokens. With a small dataset that matters.

The vectorizer sits inside a pipeline so that it is refitted on the training part of every fold.

In [ ]:
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
pipe = make_pipeline(vec, clf)

pred = cross_val_predict(pipe, X, y, groups=groups, cv=cv)
print(classification_report(y, pred))
print(confusion_matrix(y, pred))

## Experiment 2. rubert-tiny2, [CLS]

The model is frozen and only produces vectors. `[CLS]` is a token with no meaning of its own, so by the last layer its vector is built almost entirely from the other tokens, which makes it a summary of the sentence.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2')
model = AutoModel.from_pretrained('cointegrated/rubert-tiny2')

enc = tokenizer(list(X), padding=True, truncation=True, max_length=64, return_tensors='pt')
with torch.no_grad():
    out = model(**enc)

print(out.last_hidden_state.shape)

In [ ]:
cls = out.last_hidden_state[:, 0, :].numpy()

pred = cross_val_predict(clf, cls, y, groups=groups, cv=cv)
print(classification_report(y, pred))

## Experiment 3. rubert-tiny2, mean pooling

Averaging cannot be done with a plain `.mean()`: short sentences are padded, and the padding vectors would dilute the result. The attention mask zeroes them out, and the sum is divided by the number of real tokens.

In [ ]:
mask = enc['attention_mask'].unsqueeze(-1)
summed = (out.last_hidden_state * mask).sum(dim=1)
counts = mask.sum(dim=1)
mean_emb = (summed / counts).numpy()

pred = cross_val_predict(clf, mean_emb, y, groups=groups, cv=cv)
print(classification_report(y, pred))

## Results on the full dataset

| Representation | Accuracy | Macro-F1 |
|---|---|---|
| TF-IDF | 0.37 | 0.34 |
| rubert `[CLS]` | 0.38 | 0.35 |
| rubert mean pooling | 0.40 | 0.37 |

See the README for the per-class breakdown and what follows from it.